# Per-Label LoG Seed Detection

Replicates XPIWIT-style Laplacian-of-Gaussian (LoG) seed detection applied **per label region** to handle uneven intensity across a 3D fluorescence microscopy image.

Each label region is independently normalised before LoG blob detection, so dim or bright regions are treated equally.

**Outputs**
- `seeds_per_label.csv` — seed coordinates (z, y, x)
- `seeds_marked.tif` — blank image with a `1` at every seed voxel (load in napari with `add_image`, `blending='additive'`)

## Parameters

Set input paths and the LoG detection threshold here. The threshold controls sensitivity: lower values detect more (possibly noisy) seeds, higher values detect only strong peaks.

In [46]:
INPUT_IMAGE_PATH = r"D:\Thesis_images\Final_dataset\all\test\seed\item_0003_ExtractSeedBasedIntensityWindowFilter\Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_1100_ExtractSeedBasedIntensityWindowFilter_Out1_odd_only.tif"
LABEL_PATH       = r"D:\Thesis_images\Final_dataset\all\test\Brain1_1340_Cont_AiryscanSR_Fly1_CalyxL_mask.tif"
OUTPUT_CSV       = r"D:\Thesis_images\Final_dataset\all\test\seeds_per_label.csv"
OUTPUT_TIF       = r"D:\Thesis_images\Final_dataset\all\test\seeds_marked.tif"

# LoG detection threshold — lower = more seeds, higher = fewer, stronger seeds
LOG_THRESHOLD = 0.2

# Which channel to use for seed detection (0-indexed). Ignored for single-channel images.
CHANNEL_IDX = 0

## Imports

In [47]:
import numpy as np
import pandas as pd
import tifffile as tiff
from skimage.feature import blob_log
from tqdm import tqdm

## Load Image and Label Volume

Both volumes must have identical spatial dimensions `(Z, Y, X)`.

## Preprocessing — Auto-detect Axis Order and Extract Channel

Inspects the image shape against the label shape `(Z, Y, X)` to locate the channel axis automatically — no need to specify the format manually.

- **3D image** → treated as `(Z, Y, X)` directly
- **4D image** → the axis whose removal leaves the same shape as the label volume is identified as the channel axis (fallback: smallest dimension)

In [48]:
img    = tiff.imread(INPUT_IMAGE_PATH)
labels = tiff.imread(LABEL_PATH)

print(f"Image  — shape: {img.shape}, dtype: {img.dtype}, range: {img.min()}–{img.max()}")
print(f"Labels — shape: {labels.shape}, dtype: {labels.dtype}, unique IDs: {len(np.unique(labels))-1} (excl. background)")

Image  — shape: (70, 1100, 1100), dtype: uint16, range: 0–65535
Labels — shape: (70, 1100, 1100), dtype: uint16, unique IDs: 19 (excl. background)


In [49]:
if img.ndim == 3:
    img_zyx = img
    detected_format = "ZYX (no channel)"
    print("Detected format: ZYX — single channel, no extraction needed")

elif img.ndim == 4:
    # Heuristic based on dimension size:
    #   channels  → size < 5
    #   Z slices  → 5 <= size < 1024
    #   Y / X     → size >= 1024
    channel_axis = next((ax for ax, size in enumerate(img.shape) if size < 5), None)

    if channel_axis is None:
        raise ValueError(
            f"Could not detect channel axis from shape {img.shape}. "
            f"Expected one dimension to be < 5 (number of channels)."
        )

    format_labels = {0: "CZYX", 1: "ZCYX", 2: "ZYCX", 3: "ZYXC"}
    detected_format = format_labels.get(channel_axis, f"channel at axis {channel_axis}")

    img_zyx = np.take(img, CHANNEL_IDX, axis=channel_axis)
    print(f"Detected format : {detected_format}")
    print(f"Channel axis    : {channel_axis}  (size={img.shape[channel_axis]}), using channel index: {CHANNEL_IDX}")

else:
    raise ValueError(f"Unexpected image dimensionality: {img.ndim}D. Expected 3D or 4D.")

assert img_zyx.shape == labels.shape, (
    f"Spatial shape mismatch after channel extraction: "
    f"image (Z,Y,X)={img_zyx.shape} vs labels={labels.shape}"
)

print(f"Working image   : shape={img_zyx.shape}, dtype={img_zyx.dtype}")

Detected format: ZYX — single channel, no extraction needed
Working image   : shape=(70, 1100, 1100), dtype=uint16


## Per-Label, Per-Slice LoG Seed Detection

For each label region, iterates over every Z-slice where that label exists:
1. **Mask** — isolate the label pixels in this slice
2. **Skip** — if fewer than 10 pixels in the slice, skip (too small to detect)
3. **Normalise** — subtract mean and divide by std within the label pixels of this slice
4. **Detect** — run 2D `blob_log` on the normalised slice
5. **Collect** — record seed coordinates as global `(z, y, x)`

Seeds are saved identically to the 3D approach — CSV and marker TIFF with the same shape as the input volume.

In [50]:
unique_labels = np.unique(labels)
unique_labels = unique_labels[unique_labels != 0]  # skip background

all_seeds = []  # list of (z, y, x) tuples

for label_id in tqdm(unique_labels, desc="Detecting seeds per label"):
    mask = labels == label_id

    # Z range where this label exists
    z_indices = np.where(mask.any(axis=(1, 2)))[0]

    for z in z_indices:
        slice_mask = mask[z]          # 2D mask for this slice
        slice_img  = img_zyx[z].astype(np.float32)

        # Tight 2D bounding box within this slice
        yx_coords = np.argwhere(slice_mask)
        y0, x0 = yx_coords.min(axis=0)
        y1, x1 = yx_coords.max(axis=0) + 1

        patch_img  = slice_img[y0:y1, x0:x1]
        patch_mask = slice_mask[y0:y1, x0:x1]

        voxels = patch_img[patch_mask]
        if len(voxels) < 10:
            continue  # too few pixels to normalise reliably

        mean, std = voxels.mean(), voxels.std()
        if std == 0:
            continue  # uniform slice — no peak to detect

        patch_norm = np.where(patch_mask, (patch_img - mean) / std, 0.0)

        # 2D LoG blob detection on this slice
        blobs = blob_log(
            patch_norm,
            min_sigma=1,
            max_sigma=6,
            num_sigma=6,
            threshold=LOG_THRESHOLD
        )

        # Convert patch-local (y, x) back to global coordinates
        for blob in blobs:
            y = int(blob[0]) + y0
            x = int(blob[1]) + x0
            all_seeds.append((z, y, x))

print(f"\nTotal seeds detected: {len(all_seeds)}")

Detecting seeds per label: 100%|██████████| 19/19 [00:03<00:00,  4.88it/s]


Total seeds detected: 1018


## Merge Seeds

Combine all per-label seeds into a single `(N, 3)` array.

In [51]:
if len(all_seeds) == 0:
    raise RuntimeError("No seeds detected. Try lowering LOG_THRESHOLD.")

seeds = np.array(all_seeds, dtype=np.int32)  # shape (N, 3)
print(f"Seeds array shape: {seeds.shape}")
print(f"Z range: {seeds[:, 0].min()}–{seeds[:, 0].max()}")
print(f"Y range: {seeds[:, 1].min()}–{seeds[:, 1].max()}")
print(f"X range: {seeds[:, 2].min()}–{seeds[:, 2].max()}")

Seeds array shape: (1018, 3)
Z range: 2–69
Y range: 313–791
X range: 282–907


## Save Seeds as CSV

Columns: `z`, `y`, `x` — one row per detected seed.

In [52]:
df = pd.DataFrame(seeds, columns=["z", "y", "x"])
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} seeds to: {OUTPUT_CSV}")
df.head()

Saved 1018 seeds to: D:\Thesis_images\Final_dataset\all\test\seeds_per_label.csv


,z,y,x
0,2,576,554
1,2,567,553
2,3,563,560
3,3,586,551
4,3,572,544


## Save Seed Marker Image

A blank volume (same shape and dtype as the input) with a `1` placed at every seed voxel.

Load in napari on top of the raw image:
```python
viewer.add_image(seeds_marked, name='seeds', blending='additive', colormap='red')
```

In [53]:
seeds_marked = np.zeros(img.shape, dtype=img.dtype)

for z, y, x in seeds:
    seeds_marked[z, y, x] = 1

tiff.imwrite(OUTPUT_TIF, seeds_marked)
print(f"Seed marker image shape: {seeds_marked.shape}, dtype: {seeds_marked.dtype}")
print(f"Non-zero voxels: {np.count_nonzero(seeds_marked)}")
print(f"Saved to: {OUTPUT_TIF}")

Seed marker image shape: (70, 1100, 1100), dtype: uint16
Non-zero voxels: 1018
Saved to: D:\Thesis_images\Final_dataset\all\test\seeds_marked.tif
